# Arrival Dynamics 1 — Short-Term Border Arrivals

This notebook shows how observations about a small number of daily refugee arrivals at a border crossing point can be turned into short-term planning forecasts during an unfolding influx situation.

The aim is not to predict a real emergency scenario with all its complexity. The data are synthetic, and the scenario is deliberately simplified. The purpose is to make the forecasting logic visible: what can be estimated from the information available at a given moment, what remains uncertain, and how different modelling assumptions can affect operational preparedness.

In many field settings, short-term planning is driven by simple averages and recent experience. That is often reasonable as a starting point. But mean-based, backward-looking estimates can hide important uncertainty. This notebook shows how the same basic information can be strengthened with simple probabilistic logic: not to produce certainty, but to make uncertainty explicit, quantify plausible ranges, and support better preparedness decisions.

We will follow a 30-day influx episode with synthetically generated data. The full episode exists in the background, but the notebook reveals it step by step. At each decision point, the analysis uses only the observations that would have been available to humanitarian teams at that time.

The scale of arrivals has been deliberately kept low so that the logic is easier to follow. The same ideas can be applied to larger operations; the numbers would change, but the reasoning remains the same.

## Scenario

A conflict escalation in a neighbouring country leads to increasing arrivals over several weeks. We will follow one particular border crossing point. Reception and registration teams need to decide how much capacity to prepare before the full scale of the movement is known.

During the first few days, the available observations suggest that arrivals are rising. But planners do not yet know whether the increase will stabilise, continue gradually, or develop into a larger surge.

## How the analysis is organised

The notebook is organised around decision points.

At each decision point, we will:

1. show the arrival observations available at that moment;
2. make a short-term forecast for the next planning horizon;
3. describe the uncertainty around that forecast;
4. reveal what actually happened only after the forecast has been made;
5. update the planning problem as new information becomes available.

The synthetic dataset contains internal fields used to construct the episode, but those fields are not used as planning information. The analysis works only with the arrival observations that would have been visible to humanitarian teams at the relevant point in time.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt


CANONICAL_RELATIVE_PATH = Path("data/processed/arrival_dynamics_short_term_master.csv")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / CANONICAL_RELATIVE_PATH).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the project root from the current working directory.")

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from forecasting.arrival_dynamics import (  # noqa: E402
    mean_baseline_forecast,
    simulate_poisson_forecast,
    summarize_poisson_totals,
)


scenario = pd.read_csv(PROJECT_ROOT / CANONICAL_RELATIVE_PATH)

# Part I — First forecast under limited information

The first decision point occurs after five days of observations. Humanitarian teams need to prepare for the next week, but they do not yet know whether the early increase is temporary or the beginning of a larger movement.

The immediate question is simple:

**Given only the first five days of arrivals, what should planners expect for days 6–12?**

## 1. Initial observations: days 1–5

At the first decision point, only five daily arrival counts are available.

The observed arrivals are:

In [ ]:
observed_days_1_5 = scenario.loc[
    scenario["day"].between(1, 5),
    ["day", "arrivals"],
].copy()

observed_days_1_5

These first five observations show an early upward signal. Daily arrivals increase from 25 on day 1 to 69 on day 5.

That is enough to raise concern. It is not enough to know the scale of what is coming. The next week could stabilise near the current level, continue increasing gradually, or move into a sharper surge.

At this point, the forecast should be understood as a planning estimate under limited information, not as a confident prediction of the next week.

## 2. A simple point forecast

A natural first step is to use the average of the observations so far.

Using days 1–5, the observed mean is:

In [ ]:
observed_mean = observed_days_1_5["arrivals"].mean()
pd.DataFrame(
    {
        "Measure": ["Observed mean arrivals per day"],
        "Value": [observed_mean],
    }
)

This gives a simple daily forecast of **45.2 arrivals per day**.

For a seven-day planning horizon, this implies:

In [ ]:
baseline_forecast = mean_baseline_forecast(observed_days_1_5)
baseline_total = baseline_forecast["forecast_total_arrivals"].iloc[0]

pd.DataFrame(
    {
        "Forecast start day": [baseline_forecast["day"].min()],
        "Forecast end day": [baseline_forecast["day"].max()],
        "Daily forecast arrivals": [baseline_forecast["forecast_arrivals"].iloc[0]],
        "Forecast horizon days": [baseline_forecast["forecast_horizon_days"].iloc[0]],
        "Forecast total arrivals": [baseline_total],
    }
)

The deterministic forecast for days 6–12 is therefore **316.4 arrivals** in total.

This is a useful baseline because it is transparent and easy to explain. But it has an important limitation: it gives only one number.

A point forecast does not show how much arrivals might vary around the estimate. It also does not protect against a deeper problem: the early average may stop being a reasonable guide if the underlying arrival level changes.

## 3. Adding uncertainty with a Poisson forecast

The next step is to represent uncertainty around the initial rate estimate.

Here we use a simple Poisson forecast. A Poisson model is often used for count data: situations where we are counting how many events occur in a fixed period. In this case, the “events” are daily arrivals, and the fixed period is one day.

We estimate the daily arrival rate from the first five days. Then, using a few lines of code, we simulate **100,000 possible seven-day futures** for days 6–12. Each simulated future assumes that the early observed rate remains the best available estimate of the daily arrival level.

The daily arrival rate is estimated from the first five days:

In [ ]:
poisson_simulations = simulate_poisson_forecast(
    observed_days_1_5,
    n_simulations=100_000,
    seed=20240529,
)

pd.DataFrame(
    {
        "Measure": ["Estimated daily Poisson rate"],
        "Value": [observed_mean],
    }
)

The Poisson simulation keeps the same basic expectation as the point forecast, but it shows how much the seven-day total might vary if the early observed rate remained a reasonable estimate for the next week.

The simulated seven-day totals are summarised below:

In [ ]:
poisson_summary = summarize_poisson_totals(
    poisson_simulations,
    percentiles=(5, 25, 50, 75, 90, 95),
)
poisson_summary

Under this assumption, the simulated seven-day total is centred around about **316 arrivals**. The 5th to 95th percentile range is approximately **287 to 346 arrivals**.

In other words, if the early observed daily rate remained valid, most simulated futures would produce a seven-day total somewhere in that range. A total near 287 would be relatively low under this assumption, while a total near 346 would be relatively high. The point is not that arrivals will definitely fall inside this range. The point is that the range describes what ordinary random variation looks like under the initial rate assumption.

This is more informative than the point forecast alone. It tells planners that, if the early daily rate remains valid, normal random variation could still lead to somewhat lower or higher totals over the next week.

But the phrase **if the early daily rate remains valid** matters. This forecast represents uncertainty around the initial estimate. It does not account for a rapid change in the underlying arrival level.

## 4. Revealing the first forecast horizon: days 6–12

We now reveal what happened during the first forecast horizon.

The actual arrivals for days 6–12 were:

In [ ]:
actual_days_6_12 = scenario.loc[
    scenario["day"].between(6, 12),
    ["day", "arrivals"],
].copy()

actual_days_6_12

The actual seven-day total was:

In [ ]:
actual_total = actual_days_6_12["arrivals"].sum()
poisson_totals = poisson_simulations.groupby("simulation_id")["simulated_arrivals"].sum()
poisson_p95 = poisson_summary.loc[poisson_summary["statistic"] == "p95", "value"].iloc[0]
exceedance_count = int((poisson_totals >= actual_total).sum())
exceedance_rate = exceedance_count / len(poisson_totals)

forecast_comparison = pd.DataFrame(
    {
        "Measure": [
            "Deterministic baseline total",
            "Actual total",
            "Actual to baseline ratio",
            "Poisson 95th percentile total",
            "Poisson simulations at or above actual",
            "Poisson empirical exceedance rate",
        ],
        "Value": [
            baseline_total,
            actual_total,
            actual_total / baseline_total,
            poisson_p95,
            exceedance_count,
            exceedance_rate,
        ],
    }
)

forecast_comparison

In [ ]:
chart_data = pd.DataFrame(
    {
        "forecast_reference": [
            "Baseline total",
            "Poisson p95",
            "Actual total",
        ],
        "arrivals": [baseline_total, poisson_p95, actual_total],
    }
)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    chart_data["forecast_reference"],
    chart_data["arrivals"],
)

ax.set_ylabel("Seven-day arrivals")
ax.set_title("Initial forecast versus actual days 6–12 arrivals")
ax.bar_label(bars, fmt="%.0f", padding=3)
ax.set_ylim(0, max(chart_data["arrivals"]) * 1.15)

plt.tight_layout()

The result is far outside the initial forecast range.

The deterministic baseline forecast was **316.4 arrivals**. The actual total was **858 arrivals**, about **2.7 times** the baseline. It was also far above the 95th percentile of the initial Poisson forecast.

In the simulation, none of the 100,000 Poisson futures reached a total as high as the actual days 6–12 total.

This does not mean probabilistic forecasting was useless. The Poisson forecast did what it was asked to do: it described uncertainty around the early observed rate.

The problem was that the early observed rate quickly became outdated. After day 5, the arrival level changed sharply. The initial forecast failed mainly because the underlying arrival intensity increased, not because ordinary random fluctuation around the early rate was unusually large.

This distinction matters for the rest of the notebook.

A probabilistic forecast can show uncertainty around a stated assumption. But if the expected arrival level itself is changing, the forecast must be updated as new observations arrive.

At the next decision point, planners have seen days 1–12. The question is no longer whether the first five-day average is enough. The question becomes how to update the expected arrival level, and how much high-demand risk to plan for during the next high-pressure period.